# 02 · LoRA fine-tune — **SMOKE TEST** (5% of the mixture)

Disposable copy of `02_lora_finetune.ipynb` that runs the **identical code path**
on 5% of the already-built `data/vi_mix/` mixture. Purpose: prove the fine-tune
actually runs — collator shapes, LoRA attachment, VRAM headroom, checkpoint
selection, eval + benchmark plumbing — before committing to the multi-hour run.

Every output path is suffixed so this can never overwrite real results:

| real run | smoke run |
|---|---|
| `checkpoints/vi_lora` | `checkpoints/vi_lora_smoke` |
| `results/lora_predictions.csv` | `results/smoke_lora_predictions.csv` |
| `results/lora_metrics.json` | `results/smoke_lora_metrics.json` |
| `results/lora_bench_*` | `results/smoke_lora_bench_*` |

`results/bench_metrics.json` is read but never written.

**The WER numbers here are meaningless** — the point is that every cell executes.
Delete this file once the real run is going.

In [1]:
import os, random, numpy as np, torch
from dotenv import load_dotenv
load_dotenv()
assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in .env (see .env.example)"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

torch 2.8.0+cu128 | device: cuda | NVIDIA GeForce RTX 5080


## 1 · Build / load the training mixture
Reuses the shared `data/vi_asr/` viVoice cache and builds `data/vi_mix/`
(viVoice + VIVOS + Common Voice + VLSP 2020). Cache-aware: the first run
streams ~5 GB and writes 16 kHz wavs; later runs are a no-op.

In [2]:
import data_prep, mixture

# viVoice cache (also feeds the mixture) — no-op if already built
data_prep.prepare_dataset(target_hours=8.0)

# Multi-source training mixture: viVoice + VIVOS + Common Voice 17 vi + VLSP 2020.
# Cache-aware; the first build streams ~5 GB and writes 16 kHz wavs (~1 h).
mixture.prepare_mixture()

mix = data_prep.load_splits("data/vi_mix")
train, val = mix["train"], mix["val"]

# `test` stays the viVoice held-out split so the before/after against
# results/baseline_metrics.json remains apples-to-apples. It is the only test set
# here that is still speaker-disjoint AND unseen. The external benchmarks are
# evaluated separately in section 8.
test = data_prep.load_splits("data/vi_asr")["test"]

print({"train": len(train), "val": len(val), "test (viVoice)": len(test)})
print(mixture.summarize(list(train)).to_string(index=False))

# ---- SMOKE TEST: 5% of the built mixture --------------------------------- #
SMOKE_FRAC = 0.05

def _subsample(ds, frac=SMOKE_FRAC, seed=SEED):
    n = max(2, int(len(ds) * frac))     # >=2 so the collator sanity check works
    return ds.shuffle(seed=seed).select(range(n))

train, val, test = _subsample(train), _subsample(val), _subsample(test)
print("SMOKE sizes:", {"train": len(train), "val": len(val), "test": len(test)})
print(mixture.summarize(list(train)).to_string(index=False))


[data_prep] cache hit at data/vi_asr; skipping build.
[mixture] cache hit at data/vi_mix; skipping build.
{'train': 19885, 'val': 1014, 'test (viVoice)': 114}
       source     n  hours
       cmv_vi  2298  2.895
      vivoice   980  6.409
        vivos 11660 14.921
vlsp2020_100h  4947  9.849
SMOKE sizes: {'train': 994, 'val': 50, 'test': 5}
       source   n  hours
       cmv_vi 114  0.144
      vivoice  56  0.343
        vivos 599  0.770
vlsp2020_100h 225  0.473


## 2 · Load the base model + processor

In [3]:
from transformers import AutoProcessor, AutoModelForMultimodalLM
MODEL_ID = "Qwen/Qwen3-ASR-1.7B-hf"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, attn_implementation="sdpa", device_map=DEVICE,
)
print("loaded", type(model).__name__, model.dtype, model.device)

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

loaded Qwen3ASRForConditionalGeneration torch.bfloat16 cuda:0


## 3 · Attach the LoRA adapter (decoder attention only)

`q/k/v_proj` exist in both the audio encoder and the text decoder, so we match
**only** the `model.language_model` attention projections with a regex — the
audio encoder stays frozen.

In [4]:
from peft import LoraConfig, get_peft_model

# full-match regex over module names -> only the 28 decoder layers' attn projections
TARGET_RE = r"model\.language_model\.layers\.\d+\.self_attn\.(q_proj|k_proj|v_proj|o_proj)"

model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()   # required for grad checkpointing + PEFT

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM", target_modules=TARGET_RE,
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

n_lora = sum(1 for n, _ in model.named_modules() if n.endswith("lora_A.default"))
assert n_lora == 28 * 4, f"expected 112 LoRA-adapted projections, got {n_lora}"
print("LoRA-adapted projections:", n_lora)

trainable params: 6,422,528 || all params: 2,044,475,008 || trainable%: 0.3141
LoRA-adapted projections: 112


## 4 · Data collator (on-the-fly)

For each raw example we build the ASR request (audio + prompt), then append the
target transcription tokens + EOS as **labels**, masking the prompt with `-100`.
Features are computed on the fly (not serialized to disk) since the dataset is small.

In [5]:
EOS_ID = processor.tokenizer.eos_token_id
PAD_ID = processor.tokenizer.pad_token_id

def collate(features):
    ids_list, lbl_list, feats, feat_masks = [], [], [], []
    for ex in features:
        arr, _ = data_prep.read_audio(ex["audio_path"])   # 16 kHz mono
        req = processor.apply_transcription_request(audio=arr, language="Vietnamese")
        p_ids = req["input_ids"][0]
        tgt = processor.tokenizer(ex["text"], add_special_tokens=False,
                                  return_tensors="pt")["input_ids"][0]
        tgt = torch.cat([tgt, torch.tensor([EOS_ID], dtype=tgt.dtype)])
        ids = torch.cat([p_ids, tgt])
        lbl = torch.cat([torch.full((len(p_ids),), -100, dtype=torch.long), tgt])
        ids_list.append(ids); lbl_list.append(lbl)
        feats.append(req["input_features"][0])
        feat_masks.append(req["input_features_mask"][0])

    maxlen = max(len(x) for x in ids_list)
    def pad1(x, val):
        return torch.cat([x, torch.full((maxlen - len(x),), val, dtype=x.dtype)])
    input_ids = torch.stack([pad1(x, PAD_ID) for x in ids_list])
    labels = torch.stack([pad1(x, -100) for x in lbl_list])
    attn = torch.stack([
        torch.cat([torch.ones(len(x), dtype=torch.long),
                   torch.zeros(maxlen - len(x), dtype=torch.long)]) for x in ids_list])

    maxT = max(f.shape[-1] for f in feats)
    input_features = torch.stack(
        [torch.cat([f, torch.zeros(f.shape[0], maxT - f.shape[-1], dtype=f.dtype)], dim=-1)
         for f in feats])
    input_features_mask = torch.stack(
        [torch.cat([m, torch.zeros(maxT - len(m), dtype=m.dtype)]) for m in feat_masks])

    return {"input_ids": input_ids, "attention_mask": attn, "labels": labels,
            "input_features": input_features, "input_features_mask": input_features_mask}

# sanity check on 2 examples
_b = collate([train[0], train[1]])
print({k: tuple(v.shape) for k, v in _b.items()})

{'input_ids': (2, 148), 'attention_mask': (2, 148), 'labels': (2, 148), 'input_features': (2, 128, 600), 'input_features_mask': (2, 600)}


## 5 · Training configuration

In [6]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir="checkpoints/vi_lora_smoke",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=2,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    bf16=True,
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    # Make the val split actually do its job: keep the epoch that validated
    # best rather than whichever ran last.
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,             # best + latest; 1 can delete the best
    per_device_eval_batch_size=1,   # 16 GB: long clips OOM at the default 8
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,   # our collator consumes the raw columns
    dataloader_num_workers=2,
)
trainer = Trainer(model=model, args=args, train_dataset=train,
                  eval_dataset=val, data_collator=collate)
print("train steps/epoch ~", len(train) // args.gradient_accumulation_steps)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


train steps/epoch ~ 62


## 6 · Train
Watch VRAM stay < 16 GB. If OOM: raise `gradient_accumulation_steps` or lower `data_prep.MAX_SEGMENT_S` and rebuild.

In [7]:
trainer.train()
trainer.save_model("checkpoints/vi_lora_smoke")   # saves the LoRA adapter
print("peak VRAM GB:", round(torch.cuda.max_memory_allocated() / 1e9, 2))
print("adapter saved -> checkpoints/vi_lora_smoke")

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss
1,0.285972,0.442815
2,0.226411,0.460185


peak VRAM GB: 6.7
adapter saved -> checkpoints/vi_lora_smoke


## 7 · Evaluate the fine-tuned model on the test split

In [8]:
from vi_norm import normalize_vi, wer_cer   # shared with 01_eval_baseline.ipynb

model.config.use_cache = True
model.eval()
processor.tokenizer.padding_side = "left"   # required for correct batched generation
BATCH_SIZE = 8   # lower to 4 if you hit CUDA OOM

@torch.no_grad()
def transcribe_batch(rows, batch_size=BATCH_SIZE, max_new_tokens=440):
    from tqdm.auto import tqdm
    rows = list(rows)
    order = sorted(range(len(rows)), key=lambda i: rows[i]["duration"])
    hyps = [None] * len(rows)
    for s in tqdm(range(0, len(order), batch_size), desc="eval (batched)"):
        chunk = order[s:s + batch_size]
        arrs = [data_prep.read_audio(rows[i]["audio_path"])[0] for i in chunk]
        inputs = processor.apply_transcription_request(
            audio=arrs, language="Vietnamese").to(model.device, model.dtype)
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        gen = out[:, inputs["input_ids"].shape[1]:]
        for i, txt in zip(chunk, processor.decode(gen, return_format="transcription_only")):
            hyps[i] = txt
    return hyps

In [9]:
import pandas as pd
test_rows = list(test)
hyps = transcribe_batch(test_rows)   # batched, ~3x faster
rows = [{"audio_file": os.path.basename(r["audio_path"]), "audio_path": r["audio_path"],
         "channel": r["channel"], "bucket": r["bucket"], "duration": round(r["duration"], 2),
         "ref": r["text"], "hyp": h}
        for r, h in zip(test_rows, hyps)]
df = pd.DataFrame(rows)
df.to_csv("results/smoke_lora_predictions.csv", index=False)
lora_overall = wer_cer(df["ref"], df["hyp"])
lora_by = {b: wer_cer(g["ref"], g["hyp"]) for b, g in df.groupby("bucket")}
print("LoRA OVERALL:", lora_overall)
for b, m in lora_by.items():
    print(f"  {b}: WER={m['wer']:.3f} CER={m['cer']:.3f} (n={m['n']})")

eval (batched):   0%|          | 0/1 [00:00<?, ?it/s]

LoRA OVERALL: {'wer': 0.07782101167315175, 'cer': 0.04781235904375282, 'n': 5, 'wer_legacy': 0.10707070707070707, 'cer_legacy': 0.06986692015209126}
  30-60: WER=0.073 CER=0.051 (n=1)
  5-30: WER=0.082 CER=0.045 (n=4)


## 8 · Before / after comparison

In [10]:
import json, pathlib
pathlib.Path("results").mkdir(exist_ok=True)
with open("results/smoke_lora_metrics.json", "w", encoding="utf-8") as f:
    json.dump({"overall": lora_overall, "by_bucket": lora_by}, f, ensure_ascii=False, indent=2)

try:
    baseline = json.load(open("results/baseline_metrics.json"))
    comp = pd.DataFrame({
        "metric": ["WER", "CER"],
        "baseline": [baseline["overall"]["wer"], baseline["overall"]["cer"]],
        "lora": [lora_overall["wer"], lora_overall["cer"]],
    })
    comp["abs_delta"] = comp["lora"] - comp["baseline"]
    comp["rel_%"] = 100 * comp["abs_delta"] / comp["baseline"]
    print(comp.to_string(index=False))
except FileNotFoundError:
    print("Run 01_eval_baseline.ipynb first to get baseline_metrics.json for the comparison.")
    comp = None
comp

metric  baseline     lora  abs_delta     rel_%
   WER  0.059577 0.077821   0.018244 30.623533
   CER  0.030340 0.047812   0.017472 57.588956


,metric,baseline,lora,abs_delta,rel_%
0,WER,0.059577,0.077821,0.018244,30.623533
1,CER,0.030340,0.047812,0.017472,57.588956


## 9 · External benchmarks — before / after

Scores the fine-tuned adapter on the same benchmark suite as notebook 1 and
diffs it against `results/bench_metrics.json`.

> **These are no longer zero-shot.** The mixture trains on the *train* splits of
> VIVOS and Common Voice, and on 90% of the VLSP corpus (by transcript hash), so
> all three test sets are now **in-domain**. The improvement here measures
> adaptation to these datasets, not general Vietnamese ASR gains. The genuinely
> held-out, speaker-disjoint number is the viVoice `test` result in section 7.
>
> The eval rows themselves stay clean: VIVOS/Common Voice test splits are
> untouched, and the VLSP slice is hash-disjoint from everything trained on.

If the baseline file was produced before the VLSP held-out slice existed, the
cell below says so — re-run notebook 1 section 8 to refresh it.

In [11]:
import bench

processor.tokenizer.padding_side = "left"
lora_bench, lora_bench_frames = bench.run_benchmarks(
    model, processor, ["vivos", "cmv_vi", "vlsp2020_100h"],
    token=os.environ.get("HF_TOKEN"), batch_size=8, limit=16,   # SMOKE: 16 rows each
)

pathlib.Path("results").mkdir(exist_ok=True)
for name, f in lora_bench_frames.items():
    f.to_csv(f"results/smoke_lora_bench_{name}_predictions.csv", index=False)
with open("results/smoke_lora_bench_metrics.json", "w", encoding="utf-8") as f:
    json.dump(lora_bench, f, ensure_ascii=False, indent=2)
print("saved -> results/smoke_lora_bench_metrics.json")



=== vivos · htdung167/vivos-preprocessed-v2 [test] ===
    Official VIVOS test split, 760 utterances.


loading vivos: 0it [00:00, ?it/s]

transcribing (batched):   0%|          | 0/2 [00:00<?, ?it/s]

   WER=0.1122  CER=0.0667  (legacy WER=0.1122)  n=16

=== cmv_vi · fixie-ai/common_voice_17_0 [test] ===
    Common Voice 17.0 Vietnamese test split, 1274 utterances.


loading cmv_vi: 0it [00:00, ?it/s]

transcribing (batched):   0%|          | 0/2 [00:00<?, ?it/s]

   WER=0.2212  CER=0.0903  (legacy WER=0.2212)  n=16

=== vlsp2020_100h · data/vi_mix/heldout_vlsp2020_100h [test] ===
    VLSP 2020 VinBigData 100h corpus, 5% held-out slice assigned by transcript hash. Own benchmark, not PhoWhisper's T1/T2. ~96% transcript accuracy -> read the delta, not the absolute.


loading vlsp2020_100h: 0it [00:00, ?it/s]

transcribing (batched):   0%|          | 0/2 [00:00<?, ?it/s]

   WER=0.0639  CER=0.0372  (legacy WER=0.0639)  n=16
saved -> results/smoke_lora_bench_metrics.json


In [12]:
try:
    base_bench = json.load(open("results/bench_metrics.json"))
except FileNotFoundError:
    base_bench = {}
    print("No results/bench_metrics.json — run notebook 1 section 8 for the baseline.")

rows = []
for name, l in lora_bench.items():
    b = base_bench.get(name)
    if b and b.get("source") != l.get("source"):
        print(f"WARNING {name}: baseline came from {b.get('source')!r} but this run "
              f"used {l.get('source')!r}. Re-run notebook 1 section 8 — the "
              f"comparison below is not valid for this row.")
    rows.append({
        "dataset": name,
        "n": l["n"],
        "baseline WER %": round(100 * b["wer"], 2) if b else None,
        "LoRA WER %": round(100 * l["wer"], 2),
        "abs delta": round(100 * (l["wer"] - b["wer"]), 2) if b else None,
        "rel %": round(100 * (l["wer"] - b["wer"]) / b["wer"], 1) if b else None,
        "zero-shot?": "SMOKE — n differs from baseline, deltas meaningless",
    })
bench_cmp = pd.DataFrame(rows)
print()
print(bench_cmp.to_string(index=False))
bench_cmp



      dataset  n  baseline WER %  LoRA WER %  abs delta  rel %                                          zero-shot?
        vivos 16            7.19       11.22       4.04   56.2 SMOKE — n differs from baseline, deltas meaningless
       cmv_vi 16           11.18       22.12      10.94   97.9 SMOKE — n differs from baseline, deltas meaningless
vlsp2020_100h 16           13.16        6.39      -6.76  -51.4 SMOKE — n differs from baseline, deltas meaningless


,dataset,n,baseline WER %,LoRA WER %,abs delta,rel %,zero-shot?
0,vivos,16,7.19,11.22,4.04,56.2,"SMOKE — n differs from baseline, deltas meanin..."
1,cmv_vi,16,11.18,22.12,10.94,97.9,"SMOKE — n differs from baseline, deltas meanin..."
2,vlsp2020_100h,16,13.16,6.39,-6.76,-51.4,"SMOKE — n differs from baseline, deltas meanin..."
